In [64]:
# xarray to read NETCDF
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import dask as dd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
plt.rcParams['figure.dpi'] = 300

In [65]:
sst = xr.open_dataset('data/SST/sst.mnmean.nc')
sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby(['lon'])
sst_df = sst.to_dataframe().reset_index()
sst_df = sst_df[sst_df['nbnds'] == 0]
sst_df['month'] = sst_df['time'].dt.month
sst_df['year'] = sst_df['time'].dt.year
sst_df = sst_df.query('year >= 1993 and year <= 2024').reset_index().drop(['index'], axis=1)

In [66]:
chirps = xr.open_dataset('data/CHIRPS/chirps-v2.0.monthly.nc')

In [67]:
ltm_sst = sst_df.dropna().groupby(['lat', 'lon', 'month']).mean('sst').reset_index()
ltm_sst_clean = ltm_sst.drop(['time_bnds', 'nbnds', 'year'], axis=1)
ltm_sst_clean = ltm_sst_clean.rename(columns={'sst':'ltm_sst'})

In [68]:
sst_anomaly = sst_df.drop(['time_bnds', 'nbnds'], axis=1).merge(ltm_sst_clean, on=['lat', 'lon', 'month'], how='left')

In [69]:
sst_anomaly['sst_anomaly'] = sst_anomaly['sst'] - sst_anomaly['ltm_sst']

In [70]:
chirps_eastern_east_africa = chirps.sel(latitude=slice(-3.5, 8), longitude=slice(38, 50)).to_dataframe().reset_index()
chirps_eastern_east_africa['month'] = chirps_eastern_east_africa['time'].dt.month
chirps_eastern_east_africa['year'] = chirps_eastern_east_africa['time'].dt.year

In [71]:
sst_all = {}

for i in range(12):
    i += 1
    chirps_eastern_east_africa_month = chirps_eastern_east_africa.query(f'month == {i}').groupby(['month', 'year']).mean('precip').reset_index()

    chirps_eastern_east_africa_month = chirps_eastern_east_africa_month.query('year >= 1993 and year <= 2024')

    # Get tercile values
    tercile_list = chirps_eastern_east_africa_month.quantile([0.33, 0.66])['precip'].to_list()

    month_bn = chirps_eastern_east_africa_month.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    month_n = chirps_eastern_east_africa_month.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    month_an = chirps_eastern_east_africa_month.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_month = sst_anomaly.query(f'month == {i}')

    sst_df_month_bn = sst_anomaly_month[sst_anomaly_month['year'].isin(month_bn)]
    sst_df_month_n = sst_anomaly_month[sst_anomaly_month['year'].isin(month_n)]
    sst_df_month_an = sst_anomaly_month[sst_anomaly_month['year'].isin(month_an)]

    sst_df_month_dict = {'bn':sst_df_month_bn, 'n':sst_df_month_n, 'an':sst_df_month_an}

    sst_df_month = pd.concat(sst_df_month_dict.values(), keys=sst_df_month_dict.keys(), names=['tercile']).reset_index().drop(['level_1'], axis=1)

    sst_all[i] = sst_df_month.dropna().groupby(['tercile', 'lat', 'lon']).mean().reset_index()

In [77]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst'], axis=1).reset_index().set_index(['lat', 'lon', 'month', 'tercile']).to_xarray()['sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='month',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.title('Eastern East Africa')
plt.savefig(f'figures/global_sst/sst_eea.png')
plt.close()

In [73]:
chirps_southern_africa = chirps.sel(latitude=slice(-23, -15), longitude=slice(25, 34)).to_dataframe().reset_index()
chirps_southern_africa['month'] = chirps_southern_africa['time'].dt.month
chirps_southern_africa['year'] = chirps_southern_africa['time'].dt.year

In [74]:
sst_all = {}

for i in range(12):
    i += 1
    chirps_southern_africa_month = chirps_southern_africa.query(f'month == {i}').groupby(['month', 'year']).mean('precip').reset_index()

    chirps_southern_africa_month = chirps_southern_africa_month.query('year >= 1993 and year <= 2024')

    # Get tercile values
    tercile_list = chirps_southern_africa_month.quantile([0.33, 0.66])['precip'].to_list()

    month_bn = chirps_southern_africa_month.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    month_n = chirps_southern_africa_month.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    month_an = chirps_southern_africa_month.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_month = sst_anomaly.query(f'month == {i}')

    sst_df_month_bn = sst_anomaly_month[sst_anomaly_month['year'].isin(month_bn)]
    sst_df_month_n = sst_anomaly_month[sst_anomaly_month['year'].isin(month_n)]
    sst_df_month_an = sst_anomaly_month[sst_anomaly_month['year'].isin(month_an)]

    sst_df_month_dict = {'bn':sst_df_month_bn, 'n':sst_df_month_n, 'an':sst_df_month_an}

    sst_df_month = pd.concat(sst_df_month_dict.values(), keys=sst_df_month_dict.keys(), names=['tercile']).reset_index().drop(['level_1'], axis=1)

    sst_all[i] = sst_df_month.dropna().groupby(['tercile', 'lat', 'lon']).mean().reset_index()

In [76]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst'], axis=1).reset_index().set_index(['lat', 'lon', 'month', 'tercile']).to_xarray()['sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='month',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.title('Southern Africa')
plt.savefig(f'figures/global_sst/sst_sa.png')
plt.close()